# Adım 4 — Chunking ve kontrol

1. hücre `data/clean/*.txt` ve `data/legal/documents.yaml` dosyalarından `data/chunks/chunks.jsonl` üretir.  
Sonraki hücreler çıktıyı kontrol eder. Beklenen sonuçlar her hücrenin başında yazıyor.

In [ ]:
from chunky.chunk import main

main()

In [ ]:
# Beklenen: her satırda "eksik: []", tüm metadata alanları dolu, anahtar ifadeler True.
import json, re
from collections import defaultdict
from pathlib import Path

ROOT = Path.cwd().parent
chunks = [json.loads(line) for line in (ROOT / "data" / "chunks" / "chunks.jsonl").open(encoding="utf-8")]
by_id = {c["chunk_id"]: c for c in chunks}

# 1) Madde numaraları kesintisiz mi? (Madde / Ek Madde / Geçici Madde, belge başına)
groups = defaultdict(list)
for c in chunks:
    if m := re.fullmatch(r"(m|ekm|gm)(\d+)[A-ZÇ]?", c["chunk_id"].split("-", 1)[1]):
        groups[(c["doc_code"], m[1])].append(int(m[2]))
for (code, kind), nums in groups.items():
    missing = sorted(set(range(1, max(nums) + 1)) - set(nums))
    print(f"{code:8} {kind:4} {len(nums):4} madde  eksik: {missing}")

# 2) Metadata eksiksiz mi? (article_heading ve section bazı chunk'larda boş olabilir)
required = ["chunk_id", "article_id", "doc_code", "doc_title", "doc_type", "article_label",
            "source_url", "accessed", "version_date", "text"]
empty = [(c["chunk_id"], f) for c in chunks for f in required if not c.get(f)]
print("\nBoş zorunlu alan:", empty[:10] or "yok")

# 3) Tablo ve ek içerikleri doğru chunk'ta mı?
for cid, phrase in [("KTY-m100", "Otomobil (M1), (M1G), | 50 | 90 | 110 | 120"),
                    ("SAĞLIK-m10", "Turnike"), ("KTY-cetvel1", "eflektif")]:
    print(cid, phrase in by_id[cid]["text"])

In [ ]:
# Eval setindeki her atıf bir chunk'a karşılık geliyor mu?  Beklenen: "eşleşmeyen: 0"
import openpyxl

EVAL_PATH = ROOT / "docs" / "eval" / "Eval_Set_1.xlsx"   # dosya başka yerdeyse burayı değiştir
#C:\projects\chunky-502\docs\eval\Eval_Set_1.xlsx
REF = re.compile(r"(KTK|KTY|KURS|SAĞLIK|MUAYENE|SKUTER)\s+(Art\.|Additional Article|Provisional Article|Annex-?)\s*(\d+)(/[A-ZÇ](?![\w-]))?")
KIND = {"Art.": "m", "Additional Article": "ekm", "Provisional Article": "gm"}

def to_chunk_id(code, kind, no, letter):
    if kind.startswith("Annex"):                       # KTY'de ekler "N Sayılı Cetvel" adını taşır
        return next((f"{code}-{p}{no}" for p in ("cetvel", "ek", "liste") if f"{code}-{p}{no}" in by_id), f"{code}-ek{no}")
    return f"{code}-{KIND[kind]}{no}{(letter or '').strip('/')}"

ws = openpyxl.load_workbook(EVAL_PATH, read_only=True)["Questions"]
rows = list(ws.iter_rows(values_only=True))
col = rows[0].index("Legal Article Reference")
total, unmatched = 0, []
for row in rows[1:]:
    ref = row[col] or ""
    for m in REF.findall(ref):
        cid = to_chunk_id(*m)
        total += 1
        if cid not in by_id:
            unmatched.append((row[0], cid))
print(f"{total} atıf, eşleşmeyen: {len(unmatched)}", unmatched)

In [ ]:
# Bilgi: en uzun chunk'lar (alt-parçalama kararı embedding modeli seçilince verilecek, AD-08)
for c in sorted(chunks, key=lambda c: -len(c["text"]))[:10]:
    print(f'{c["chunk_id"]:16} {len(c["text"]):6} karakter  {c["article_label"]}')
print("\nToplam:", len(chunks), "chunk")